# Feature engineering

Objective: making a new feature to characterize the karyotype of the patient, and identify the favorable and unfavorable karyotypes based on the literature.

## Importing modules

In [ ]:
import numpy as np
import pandas as pd
from leukemia.data import loader
import os

## Importing data

In [ ]:
X_clinical = pd.read_csv('../data/raw/X_train/clinical_train.csv')
Y = pd.read_csv('../data/raw/target_train.csv')

clinical_dataset = loader.join_clinical_target(X_clinical, Y)
clinical_dataset, n_dropped = loader.drop_missing_target(clinical_dataset)

print(f"Number of dropped targets: {n_dropped}")
clinical_dataset.head()

## 1. CYTOGENETICS feature

During the preprocessing we have simplified the cytogenetics feature. \
We have made 4 flags:
- Normal
- Abnormal
- Complex
- Missing

We considered all abnormal karyotypes equivalent, but it is not the case. The litterature suggest that some abnormal karyotypes favorable, intermediate ande adverse. \
Schanz J, Tüchler H, Solé F, Mallo M, Luño E, Cervera J, Granada I, Hildebrandt B, Slovak ML, Ohyashiki K, Steidl C, Fonatsch C, Pfeilstöcker M, Nösslinger T, Valent P, Giagounidis A, Aul C, Lübbert M, Stauder R, Krieger O, Garcia-Manero G, Faderl S, Pierce S, Le Beau MM, Bennett JM, Greenberg P, Germing U, Haase D. New comprehensive cytogenetic scoring system for primary myelodysplastic syndromes (MDS) and oligoblastic acute myeloid leukemia after MDS derived from an international database merge. J Clin Oncol. 2012 Mar 10;30(8):820-9. doi: 10.1200/JCO.2011.35.6394. Epub 2012 Feb 13. PMID: 22331955; PMCID: PMC4874200. https://pmc.ncbi.nlm.nih.gov/articles/PMC4874200/

The goal is to crate new flags, that match teh categories identified by Schanz and al. :
- Normal
- Favorable
- Intermediate
- Adverse
- Complex
- Missing \
\
The coresponding karyotypes are :
- Very good: del(11q) -Y
- Good: Normal, del(5q) (single and double), del(12p), del(20q)
- Intermediate: del(7), +8, i(17q), +19, any double, any other single
- Poor : single: inv(3), t(3q), del(3q), -7 ; double: -7 del(7q) complex = 3 anomaly
- Very poor: complex > 3 anomaly \
\
Counting the number of anomaly: we are using a proxy: the number of "," minus one in the longest clone. Indeed, a coma separate the abnormalities. We subtract the coma that separate the number of chromosomes and the rest of the karyotype.

Blueprint of the notation :
- del(5q) can be del(5q), del(5)(q5), del5q, 5q- --> del\({chrom}({arm}|\({arm})|del{chrom}{arm}|{chrom}{arm}-

In [ ]:
def patern_del_arm(chrom: str, arm: str) -> str:
    """Cover del(20q), del(20)(q12) and free labels del20q/20q-."""
    return rf"del\({chrom}({arm}|\({arm}|\)\({arm})|del{chrom}{arm}|{chrom}{arm}-"

In [ ]:
def count_abnormalities (cyto:str) -> int:
    """
    Count the number of abnormalities in the most abnormal clone of the karyotype
    :param cyto: the karyotype of the patient
    :return: the number of abnormalities
    """
    return max(c.count(",") -1 for c in cyto.split("/"))

In [ ]:
def pronostic_cyto (clinical_dataset:str) -> pd.DataFrame:
    """
    Add the new feature with categories indicating the pronostic per karyotype
    :param clinical_dataset: the clinical dataset
    :return: the dataset with the new feature and without CYTOGENETICS
    """
    cyto = clinical_dataset['CYTOGENETICS']
    out = clinical_dataset.copy()
    # Number of abnormaalities
    n_abn = cyto.fillna("").map(count_abnormalities)

    # Masks
    is_missing = cyto.isna()
    is_normal = cyto.str.match(r'46,(xx|xy)(\[\d+\])?$', case=False, na=False)
    is_complex = cyto.str.match ('complex', case=False, na=False) | (n_abn == 3)
    is_very_good = cyto.str.contains(rf'({patern_del_arm('11','q')}|-Y)', case=False, na=False)
    is_good = cyto.str.contains(rf'({patern_del_arm('5','q')}|{patern_del_arm('12','p')}|{patern_del_arm('20','q')})' , case=False, na=False)
    is_intermediate = cyto.str.contains(rf'({patern_del_arm('7','q')}|\+8|i\(17(q|\(q)|\+19)', case=False, na=False)
    is_poor = cyto.str.contains(rf'(inv\(3\)|t\(3;3|{patern_del_arm('3','q')}|-7)' , case=False, na=False) | cyto.str.match ('complex', case=False, na=False) | (n_abn == 3)
    is_very_poor = (n_abn > 3) | cyto.str.contains('>3abnormalities', case=False, na=False)

    conditions = [is_missing, is_normal, is_very_poor, is_poor, is_good, is_very_good]
    cyto_categories = ["MISSING", "NORMAL", "VERY_POOR", "POOR", "GOOD", "VERY_GOOD"]
    out["CYTO_LVL"] = np.select(conditions, cyto_categories, default="INTERMEDIATE")
    out.drop('CYTOGENETICS', axis=1, inplace=True)
    return out

clinical_dataset = pronostic_cyto(clinical_dataset)
clinical_dataset["CYTO_LVL"].value_counts()


## Transforming the clinical dataset

In [ ]:
from leukemia.features import preprocessor

clinical_dataset = preprocessor.add_missing_indicator(clinical_dataset)
clinical_dataset = preprocessor.log_transform(clinical_dataset)
clinical_dataset.columns




In [ ]:
os.makedirs('../data/processed', exist_ok=True)
clinical_dataset.to_csv('../data/processed/clinical_train_feature_engin.csv', index=True)

In [ ]:
test = pd.read_csv('../data/processed/clinical_train_feature_engin.csv', index_col='ID')

In [ ]:
test['CYTO_LVL']